<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_3_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🤔 What Even Is an LLM? (Read This First)

Before we touch prompts, let's recall that we all have the same mental model.

**LLM = Large Language Model.** Think of it as a very sophisticated autocomplete engine — but one that was trained on essentially the entire internet, millions of books, research papers, and code repositories.

Here's the simplest mental model:

```
You write text  →  LLM predicts the best next words  →  You read the response
```

That's it. There's no magic. No thinking. No feelings. The LLM has learned statistical patterns from billions of text samples, so it's *very good* at predicting what a helpful, coherent response looks like.

**Examples you may have heard of:**
| Model | Made By | You've Probably Used It As |
|-------|---------|--------------------------|
| GPT-4o | OpenAI | ChatGPT |
| Claude | Anthropic | Claude.ai |
| Gemini | Google | Google AI Studio / Bard |
| Llama | Meta | Open-source, many apps |

**Why does this matter for prompting?** Because the LLM isn't *thinking* — it's *pattern-matching*. The clearer and more structured your input, the better it can match the right pattern. That's literally what prompt engineering is.

---
# 🏗️ Part 1: Anatomy of a Prompt

## The Employee Analogy

Imagine you just hired a brilliant new employee. On Day 1 you give them:

1. **The Employee Handbook** — company values, their role, tone of voice, what they can and can't do
2. **Today's task** — 'Please write a summary of this client email'
3. **Their previous replies** — so they remember the conversation so far

That's exactly the three-part structure of every AI conversation:

| Employee World | AI World | Technical Name |
|---------------|----------|----------------|
| Employee handbook | Background instructions set before conversation | **System Message** |
| Today's task | What you type | **User Message** |
| Previous replies | What AI has said so far | **Assistant Message** |

---

## The Three Roles in Detail

### 🎛️ The System Message — Your AI's Personality Blueprint

The system message runs *before* any conversation starts. It shapes *who* the AI is and *how* it behaves.

A **strong** system message has five ingredients:

| Ingredient | What It Does | Example |
|-----------|--------------|---------|
| **Persona** | Who the AI is | 'You are a senior customer support agent at Zepto' |
| **Tone** | How it sounds | 'Respond in a warm, friendly, professional tone' |
| **Rules** | What it can/can't do | 'Never discuss competitor pricing' |
| **Format** | How answers look | 'Always reply in bullet points under 100 words' |
| **Context** | Background facts | 'Zepto is a 10-minute grocery delivery app in India' |

### 📍 Where Do You Put the System Message?

- **Claude.ai Projects:** Add it in the 'Project Instructions' box — it applies to every chat in that project
- **ChatGPT Custom Instructions:** Settings → Customize ChatGPT
- **Free UI (no dedicated field):** Just start your first message with 'For this conversation, you are...'
- **Via API (for developers):** Pass it as `{"role": "system", "content": "..."}`

---

### 🔴 Weak System Message vs 🟢 Strong System Message

**🔴 Weak:**
```
You are a helpful assistant for an e-commerce company.
```

**Expected AI behaviour:** Generic, inconsistent, might apologise for things it shouldn't, no clear personality.

---

**🟢 Strong:**
```
You are Zara, a senior customer support agent at Zepto — India's 10-minute grocery delivery app.

TONE: Warm, empathetic, solution-first. Use simple English. Avoid corporate jargon.

YOUR JOB:
- Resolve order complaints (delays, missing items, wrong items)
- Escalate refund requests above ₹500 to the human team
- Never promise delivery times you cannot confirm

FORMAT: Reply in 3 sentences or fewer unless the customer asks for detail.

CONTEXT: Zepto operates in 25+ Indian cities. Our SLA for delivery is 10 minutes.
```

**Expected AI behaviour:** Introduces itself as Zara, stays concise, escalates properly, never invents delivery windows.

---

## 💬 The User Message — Your 4-Part Formula

The best user messages follow this four-part structure:

```
[TASK]    — What do you want the AI to do?
[CONTEXT] — Why? What's the situation?
[DATA]    — What information should it use?
[FORMAT]  — How should the response look?
```

You don't need all four every time — but the more you include, the better the output.

### 🔴 Weak User Message
```
Write an email about our product launch.
```
**Expected AI response:** A completely generic product launch email with placeholder text like '[Product Name]' and '[Company Name]'. Useless.

---

### 🟢 Strong User Message
```
[TASK] Write a product launch announcement email.

[CONTEXT] We are Nykaa, India's leading beauty e-commerce platform.
We are launching 'GlowFast' — a new 2-hour delivery service for skincare products.
Target audience: existing Nykaa customers aged 22–35, urban, beauty-conscious.

[DATA]
- Launch date: 15 July 2025
- Available in: Mumbai, Delhi, Bangalore, Hyderabad
- Key benefit: Freshly restocked skincare before a big event
- Offer: First GlowFast delivery free (use code GLOWFREE)

[FORMAT]
- Subject line options: Give me 3
- Body: 150 words max
- End with a clear call to action
- Tone: Excited but sophisticated
```

**Expected AI response:**
```
Subject Line Options:
1. Your glow, delivered in 2 hours ✨
2. GlowFast is here — because great skin can't wait
3. Mumbai, Delhi, Bangalore & Hyderabad: Meet GlowFast

---

Dear Nykaa Fam,

Great skin shouldn't have a next-day delivery window.

Introducing GlowFast — Nykaa's new 2-hour skincare delivery service,
now live in Mumbai, Delhi, Bangalore, and Hyderabad.

Whether you're prepping for a last-minute date or your serum ran out,
we'll get your essentials to you before it matters.

🎁 Your first GlowFast delivery is on us. Use code GLOWFREE at checkout.

[Shop Now → Nykaa.com/GlowFast]
```

---

## 🤖 The Assistant Message & Prefilling

The assistant messages are what the AI has said in the conversation so far. You normally don't write these — but there's one powerful exception:

### ✨ Assistant Prefilling (Claude-Specific)

Claude allows you to *start* the assistant's response for it. This is called **prefilling**. When you begin the assistant reply with `{`, Claude is forced to continue writing JSON — no markdown wrapper, no commentary, just pure JSON.

```
User: Extract the order details from this text as JSON:
'Raj ordered 2 units of product SKU-789 on 12 July at ₹450 each.'

Assistant: {    ← You start this. Claude MUST continue as valid JSON.
```

**Claude's forced output:**
```json
{
  "customer": "Raj",
  "quantity": 2,
  "sku": "SKU-789",
  "date": "12 July",
  "price_per_unit": 450,
  "total": 900
}
```

---

## 🏷️ Delimiters — Keeping Data Separate from Instructions

A **delimiter** is a visual fence that separates different parts of your prompt. Without them, the AI can confuse your data with your instructions.

| Delimiter Type | Looks Like | Best For |
|---------------|-----------|---------|
| Triple backticks | ` ``` ` | Code, raw text, pasted documents |
| XML tags | `<data>...</data>` | Structured data (Claude's favourite — more in Part 2) |
| Markdown headers | `## Section` | Long prompts with multiple sections |
| Square brackets | `[INPUT]` | Quick labelling |

**Example:**
```
Translate the following customer review into Hindi.

<review>
The delivery was incredibly fast and the packaging was perfect.
I'll definitely order again!
</review>
```

---

## 🔁 Quick Recap — Part 1

| Concept | One-Line Summary |
|---------|-----------------|
| System Message | The employee handbook — sets persona, tone, rules, format, context |
| User Message | Your request — use Task + Context + Data + Format |
| Assistant Message | What AI said — normally auto, but you can prefill for JSON |
| Delimiters | Visual fences that separate data from instructions |

## 🧪 Try This Yourself

Open Claude.ai. Create a new Project. In the 'Project Instructions' box, write a system message for a **Razorpay payment support bot**. Include: persona, tone, two rules it must follow, and one formatting rule. Then chat with it.

---
# ✨ Part 2: The 6 Fundamentals Every Beginner Must Know
Master these and you're already ahead of 80% of people using AI.

---

## Fundamental 1: Be Clear and Direct 🎯

**The Rule:** If a human would be confused by your instruction, the AI definitely will be.

### 🔴 Unclear
```
Tell me about Infosys.
```
**Expected response:** A 500-word Wikipedia-style biography of Infosys covering history, revenue, offices, leadership, and products. Probably not what you wanted.

### 🟢 Clear and Direct
```
In exactly 3 bullet points, summarise the biggest competitive risks Infosys
faces in 2025 compared to Accenture. Each bullet max 20 words.
```
**Expected response:**
```
• Accenture has deeper AI consulting partnerships with hyperscalers (AWS, Azure, Google Cloud).
• Infosys relies more heavily on cost arbitrage; Accenture commands higher margins on strategy work.
• Accenture's global brand recognition makes enterprise client acquisition easier outside Asia.
```

---

## Fundamental 2: Assign a Role 🎭

**The Rule:** Starting with 'You are a [specific expert]' dramatically changes the quality and style of the response.

**Why it works:** The LLM has seen millions of texts written *by* senior lawyers, doctors, engineers. When you trigger that pattern, you get that quality of output.

### 🔴 No Role
```
Review this contract clause: 'Either party may terminate this agreement
with 7 days written notice.'
```
**Expected response:** A generic, cautious answer that says something like 'this seems standard, but consult a lawyer.'

### 🟢 With Role
```
You are a senior corporate lawyer specialising in Indian contract law with
15 years of experience reviewing SaaS vendor agreements.

Review this clause critically:
'Either party may terminate this agreement with 7 days written notice.'

Flag any risks for a startup founder signing this as a customer.
```
**Expected response:**
```
This clause is unusually vendor-friendly and carries three risks for a startup customer:

1. RISK — No cure period: 7 days gives you no time to fix a dispute before
   termination. Standard contracts give 30 days to remedy a breach.

2. RISK — No data retrieval window: If the vendor terminates, 7 days may not
   be enough to export your data, especially for large datasets.

3. RISK — Symmetric on paper, asymmetric in practice: The vendor likely has
   this clause with 100s of customers and can terminate efficiently.
   You have one vendor relationship and will struggle to switch in 7 days.

RECOMMENDATION: Negotiate to 30-day notice with a 15-day data export window post-termination.
```

---

## Fundamental 3: Use XML Tags 🏷️

**The Rule:** Claude was **specifically trained** on XML-tagged prompts. This is Anthropic's official recommendation — not a hack or trick.

XML tags make it unmistakable to Claude where your instructions end and your data begins.

### 🔴 Without XML Tags
```
Translate this to Hindi. The text is: The payment was successful.
Please note that 'payment' should be translated as 'bhugtan' not 'payment'.
```
**Problem:** Claude might confuse 'please note' as part of the text to translate.

### 🟢 With XML Tags
```
Translate the text inside <translate> tags to Hindi.

<instructions>
- Use formal Hindi (आप form, not तुम)
- Translate 'payment' as 'भुगतान' (bhugtan), not the English word
- Output only the translated text, nothing else
</instructions>

<translate>
The payment was successful. Your order will be delivered within 2 hours.
</translate>
```
**Expected response:**
```
भुगतान सफल रहा। आपका ऑर्डर 2 घंटे के भीतर डिलीवर किया जाएगा।
```

**Common XML tags used with Claude:**
| Tag | Use Case |
|-----|---------|
| `<instructions>` | Your rules and directives |
| `<data>` | The raw data to process |
| `<example>` | A sample input/output pair |
| `<context>` | Background information |
| `<thinking>` | Ask Claude to reason before answering |
| `<output>` | Where Claude should write its final answer |

---

## Fundamental 4: Specify Output Format 📐

**The Rule:** Tell Claude *exactly* what the output should look like. Don't make it guess.

### 🔴 No Format Specified
```
Compare React, Vue, and Angular for a startup.
```
**Expected response:** Three paragraphs of prose that are hard to scan quickly.

### 🟢 Format Specified
```
You are a senior frontend architect. Compare React, Vue, and Angular for an
early-stage Indian startup with a 3-person engineering team.

Output a markdown table with these columns:
Framework | Learning Curve | Ecosystem Size | Best For | Avoid If

Then add one sentence of final recommendation.
```
**Expected response:**
```
| Framework | Learning Curve | Ecosystem Size | Best For | Avoid If |
|-----------|---------------|----------------|----------|----------|
| React | Medium | Massive | SPAs, lots of UI flexibility | Team dislikes JSX |
| Vue | Low | Medium | Quick onboarding, readable code | Need large enterprise support |
| Angular | High | Large | Enterprise apps, strict structure | Small teams, fast iteration |

**Recommendation:** For a 3-person Indian startup, go with Vue — it has the gentlest
learning curve and lets a small team move fast without sacrificing structure.
```

---

## Fundamental 5: Allow 'I Don't Know' 🛑

**The Rule:** If you don't explicitly allow Claude to say 'I don't know', it will sometimes *make something up* (hallucinate). This is one of the most important safety techniques.

### 🔴 Without Escape Hatch
```
What was Bhutan's exact GDP in Q2 2024?
```
**Risky response:** Claude might confidently give you a number that sounds right but isn't verifiable.

### 🟢 With Escape Hatch
```
What was Bhutan's exact GDP in Q2 2024?

If you do not have reliable data for this, say 'I don't have reliable data
for this specific figure' rather than estimating.
```
**Expected response:**
```
I don't have reliable quarterly GDP data for Bhutan broken down by Q2 2024.
Bhutan's annual GDP is approximately USD 2.5 billion (2023 estimate), but
I cannot give you a verified Q2 figure. For accurate data, check the
Royal Monetary Authority of Bhutan or World Bank's Bhutan country page.
```

---

## Fundamental 6: Iterate — Prompting is a Conversation 🔄

**The Rule:** Your first prompt is a draft. Great outputs come from 2–3 rounds of refinement.

### Round 1 — First Draft
```
Write an email to our team announcing we are switching from Slack to Microsoft Teams.
```
**Response:** A decent but generic announcement email.

### Round 2 — Add Context
```
Good, but our team are engineers who love Slack and will be unhappy.
Acknowledge their frustration and explain the business reason (cost savings of ₹8L/year).
Keep it under 150 words.
```
**Response:** A more empathetic email that pre-empts frustration.

### Round 3 — Refine Further
```
Perfect. Now add a one-line FAQ at the bottom answering:
'Will our Slack message history be migrated?'
Answer: Yes, 90 days of history will be imported.
```
**Response:** Final email with FAQ — ready to send.

---

## 🔁 Quick Recap — Part 2

| # | Fundamental | One-Line Memory Trick |
|---|------------|----------------------|
| 1 | Be Clear & Direct | If a human would be confused, AI will be too |
| 2 | Assign a Role | 'You are a senior X' unlocks expert-quality output |
| 3 | Use XML Tags | Claude was trained on these — they're not a hack |
| 4 | Specify Format | Tables, JSON, word counts — tell it exactly |
| 5 | Allow 'I don't know' | Prevents hallucination on unknown facts |
| 6 | Iterate | First prompt is a draft. Refine 2–3 times. |

## 🧪 Try This Yourself
Take any email you wrote this week. Paste it into Claude with this prompt: *'You are a communication coach. Review this email for clarity, tone, and length. Apply Fundamental 1 (Be Clear and Direct) and suggest exactly 3 improvements.'*

---
# 🎯 Part 3: Few-Shot Prompting

## What Is Few-Shot Prompting?

**Few-shot prompting** means giving the AI examples of what you want before asking it to do the real task. It's teaching by demonstration.

| Technique | Examples Given | Best For |
|-----------|---------------|---------|
| **Zero-shot** | 0 | Simple, clear tasks |
| **One-shot** | 1 | When you have one good example |
| **Few-shot** | 2–5 | Pattern-sensitive tasks, classification, formatting |
| **Many-shot** | 6+ | Complex patterns, rare edge cases |

---

## Example 1: Sentiment Classification

### 🔴 Zero-Shot (Inconsistent)
```
Classify the sentiment of this review:
'The app kept crashing but the delivery guy was super helpful.'
```
**Problem:** AI might say 'Mixed', 'Neutral', or write a paragraph. You can't process this at scale.

### 🟢 Few-Shot (Consistent + Controllable)
```
Classify customer reviews as POSITIVE, NEGATIVE, or MIXED.
Return only the label — nothing else.

<examples>
Review: 'The biryani was amazing and delivery was in 8 minutes!' → POSITIVE
Review: 'Wrong order delivered and customer care was rude.' → NEGATIVE
Review: 'Food was great but took 35 minutes instead of 10.' → MIXED
</examples>

<review>
The app kept crashing but the delivery guy was super helpful.
</review>
```
**Expected response:**
```
MIXED
```

---

## Example 2: Brand Voice Matching

This is where few-shot prompting gets *magical* — the AI learns your brand's personality from examples alone.

```
Write a product description for 'Nykaa Rose Glow Face Mist' in the same
voice as these existing Nykaa descriptions.

<examples>
Product: Nykaa Naturals Vitamin C Serum
Description: Wake up your skin with the glow it's been craving.
Packed with pure Vitamin C, this serum fades dark spots, evens your skin
tone, and delivers radiance you can actually see — in as little as 4 weeks.

Product: Nykaa SKINgenius Moisturiser
Description: Your skin, but smarter. This genius formula reads what your
face needs and delivers exactly that — hydration that holds all day,
with a finish that looks like you just stepped out of a facialist's chair.
</examples>

Now write for: Nykaa Rose Glow Face Mist (a hydrating mist with rose water
and hyaluronic acid for instant refresh)
```
**Expected response:**
```
One spritz and your skin remembers how good it feels to be refreshed.
This rose-kissed mist pairs the ancient glow-secret of rose water with
hyaluronic acid — because your skin deserves a drink that lasts.
Spritz. Glow. Repeat.
```
The AI matched Nykaa's poetic, second-person, punchy voice — without being told to.

---

## Example 3: Few-Shot + XML Tags Combined

Combining few-shot examples *inside* XML tags gives you maximum control:

```
Extract entities from customer support messages.
Return JSON only.

<examples>
<example>
<input>Hi, my name is Priya and I ordered a mixer grinder (Order #Z-4821) on
10th July but it hasn't arrived yet. Please help.</input>
<output>{"name": "Priya", "order_id": "Z-4821", "issue": "delivery delay", "date": "10th July"}</output>
</example>
<example>
<input>This is Raman. My UPI payment of ₹1,299 to Swiggy failed but money
was deducted. Transaction ID TXN-9920.</input>
<output>{"name": "Raman", "transaction_id": "TXN-9920", "issue": "payment failure", "amount": 1299}</output>
</example>
</examples>

<input>
Hello, I am Kavitha. My subscription to Hotstar Premium (Order ORD-77231)
was charged twice this month — ₹299 each time.
</input>
```
**Expected response:**
```json
{"name": "Kavitha", "order_id": "ORD-77231", "issue": "duplicate charge", "amount_per_charge": 299, "total_charged": 598}
```

---

## 5 Rules for Picking Good Examples

1. **Representative** — Examples should cover the most common cases, not just easy ones
2. **Diverse** — Show variety; don't use 3 examples that all look the same
3. **Accurate** — Wrong examples teach wrong patterns
4. **Consistent format** — Every example must follow the same input → output structure
5. **Edge cases last** — Put unusual examples at the end, common ones first

---

## 🔁 Quick Recap — Part 3

| Shot Type | When to Use |
|-----------|------------|
| Zero-shot | Simple, clear tasks where format doesn't matter |
| Few-shot | Consistent output needed, pattern or voice matching, classification |
| Many-shot | Complex patterns, production-grade pipelines |

## 🧪 Try This Yourself
Think of something repetitive you do: writing status update emails, classifying support tickets, tagging content categories. Write 3 examples of input → output. Now give them to Claude as a few-shot prompt and test with a new input.

---
# 🧠 Part 4: Chain-of-Thought Prompting

## What Is Chain-of-Thought (CoT)?

**Chain-of-Thought prompting** asks the AI to *show its reasoning* before giving a final answer. Instead of jumping to a conclusion, it thinks out loud — step by step.

**Analogy:** It's the difference between a student who just writes '42' as their maths answer vs one who shows all their working. The one who shows working catches more mistakes.

---

## The Research Behind It (Fact-Checked)

**Wei et al. (2022) — Chain-of-Thought Prompting Elicits Reasoning in Large Language Models (Google)**

This landmark paper showed that asking models to reason step-by-step dramatically improved performance on maths and logic tasks. On the GSM8K math benchmark, PaLM 540B with few-shot chain-of-thought reached **58% accuracy**, compared to much lower scores with standard prompting.

> 📄 Full paper: https://arxiv.org/abs/2201.11903

---

**Kojima et al. (2022) — Large Language Models Are Zero-Shot Reasoners (University of Tokyo + Google)**

This team made an even simpler discovery: you don't even need examples. Just adding the phrase **'Let's think step by step'** to any prompt improved accuracy dramatically:

| Benchmark | Before 'Let's think step by step' | After |
|-----------|----------------------------------|-------|
| MultiArith | 17.7% | 78.7% |
| GSM8K | 10.4% | 40.7% |

> 📄 Full paper: https://arxiv.org/abs/2205.11916

---

## ⚠️ 2025 Update: CoT Has Minimal Benefit on Modern Reasoning Models

> **Important nuance for 2025 and beyond:**

Research from Wharton (2025) found that manually asking for chain-of-thought has **minimal additional benefit** on reasoning-optimised models like OpenAI o1/o3 and **Claude's Extended Thinking** mode. Why? These models *already reason internally* before giving you an answer. You're essentially asking them to reason twice.

| Model Type | Does CoT prompting help? |
|-----------|------------------------|
| Standard models (Claude Haiku, GPT-4o) | ✅ Yes — significant improvement |
| Reasoning models (o1, o3, Claude Extended Thinking) | ⚠️ Minimal — they already reason internally |

**Practical rule:** Use CoT on standard models. For o1/o3/Claude Extended Thinking, just give a clear question and let the model handle the reasoning.

---

## Three Forms of Chain-of-Thought

### Form 1: Zero-Shot CoT — The Magic Phrase

```
A fruit seller starts the day with 120 apples. She sells 35 in the morning,
gets a delivery of 50 more, then sells 40 in the afternoon.
How many apples does she have left?

Let's think step by step.
```
**Expected response:**
```
Step 1: Start with 120 apples.
Step 2: Sell 35 in the morning → 120 - 35 = 85 apples.
Step 3: Receive delivery of 50 → 85 + 50 = 135 apples.
Step 4: Sell 40 in the afternoon → 135 - 40 = 95 apples.

The fruit seller has 95 apples left.
```

---

### Form 2: Few-Shot CoT — Teach the Format First

```
<example>
Question: If I pay ₹250 for a shirt and sell it for ₹320, what is my profit %?
Reasoning: Profit = 320 - 250 = ₹70. Profit % = (70 / 250) × 100 = 28%.
Answer: 28% profit.
</example>

Question: A retailer buys 50 units at ₹180 each and sells 40 units at ₹250
each. The remaining 10 units are unsold. What is the profit or loss?

Reasoning:
```
**Expected response:**
```
Reasoning:
Total cost = 50 × ₹180 = ₹9,000
Revenue from sold units = 40 × ₹250 = ₹10,000
Unsold units have no revenue: 10 × ₹0 = ₹0
Total revenue = ₹10,000
Profit = ₹10,000 - ₹9,000 = ₹1,000

Answer: ₹1,000 profit.
```

---

### Form 3: Structured CoT — Business Decisions

CoT isn't just for maths. It's excellent for complex business reasoning:

```
You are a startup advisor. Should a Bangalore-based B2B SaaS company
(30 employees, ₹2Cr ARR) expand to the UAE market right now?

Think through this step by step:
1. First, assess their current maturity
2. Then, evaluate the UAE opportunity
3. Then, identify the top 2 risks
4. Finally, give a clear YES/NO/WAIT recommendation with one condition
```
**Expected response:**
```
Step 1 — Current Maturity:
₹2Cr ARR (~$240K USD) with 30 employees is early-stage. Product-market
fit is likely proven but international infrastructure (billing, support,
legal entity) is probably not in place.

Step 2 — UAE Opportunity:
UAE is a strong B2B market — high willingness to pay, English-speaking
business culture, and it acts as a gateway to Saudi Arabia and the wider GCC.
Many Indian SaaS companies (Zoho, Freshworks) have used UAE as their first
international market.

Step 3 — Top 2 Risks:
1. BANDWIDTH RISK: Expansion requires a dedicated founder/BD head on the
   ground. At 30 people, pulling that resource could hurt the core India business.
2. SALES CYCLE RISK: UAE enterprise deals often take 6–12 months and require
   in-person relationship building — cash-intensive for a ₹2Cr ARR company.

Step 4 — Recommendation: WAIT (6–12 months)
Condition: Expand only after reaching ₹5Cr ARR, which provides the runway
to fund a 12-month UAE pilot without threatening India operations.
```

---

## 🙈 Hidden CoT with `<thinking>` Tags

Sometimes you want Claude to reason internally but *not* show its working to the end user — just the clean final answer. Use `<thinking>` tags:

```
Answer this support question. First, reason through it inside <thinking> tags.
Then give only the clean, customer-facing answer after </thinking>.

Customer question: 'I placed an order 3 days ago and it still says Processing.
Is this normal?'
```
**Expected response:**
```
<thinking>
3 days in 'Processing' is unusually long. Normal processing is 24–48 hours.
This could be: (1) payment hold, (2) inventory issue, (3) system error.
I shouldn't alarm the customer but I should escalate urgency slightly.
</thinking>

Hi there! 3 days in Processing status is longer than our usual 24–48 hour
turnaround, so I'd like to look into this for you right away. Could you
share your order number? I'll prioritise this with our fulfilment team.
```

---

## ⛔ When NOT to Use CoT

| Situation | Why CoT Doesn't Help |
|-----------|---------------------|
| Simple factual lookups | 'What's the capital of India?' doesn't need reasoning steps |
| Creative writing | Showing reasoning kills creative flow |
| Short classifications | 'Is this SPAM or NOT SPAM?' — just decide |
| Reasoning models (o1, Claude ET) | They already reason internally |

---

## 🔁 Quick Recap — Part 4

| CoT Type | Phrase to Use | Best For |
|----------|--------------|---------|
| Zero-shot | 'Let's think step by step' | Quick boost on any reasoning task |
| Few-shot | Show worked examples first | Complex patterns you want reproduced |
| Structured | 'Step 1: ... Step 2: ...' | Business decisions, multi-factor analysis |
| Hidden (`<thinking>`) | `<thinking>` tags | Reasoning internally, clean output for users |

## 🧪 Try This Yourself
Take a decision you're currently facing at work (technology choice, vendor evaluation, project prioritisation). Write a structured CoT prompt that makes Claude reason through it in 4 explicit steps.

---
# 📦 Part 5: Structured JSON Output

## Why Does JSON Matter?

When you're building a real application — not just chatting — you need the AI's output in a format your code can *read*. JSON is the universal language of APIs and applications.

**The problem:** By default, Claude is a conversational AI. It *wants* to add commentary, use markdown headers, add disclaimers. For apps, you need clean JSON — nothing else.

---

## 4 Failure Modes and How to Fix Them

| Failure Mode | What Happens | Fix |
|-------------|-------------|-----|
| Markdown wrapping | Output is wrapped in ```json ... ``` — breaks JSON.parse() | 'Return JSON only. No code blocks, no backticks.' |
| Commentary | AI adds 'Here is the JSON:' before the data | 'Output only the JSON object. No preamble.' |
| Missing keys | AI skips a field when it has no data | 'Use null for missing fields. Never omit a key.' |
| Wrong types | Numbers as strings: `"price": "450"` | 'price must be a number, not a string.' |

---

## Product Data Extraction Example

```
Extract product information from the text below.
Return ONLY a valid JSON object — no markdown, no commentary, no backticks.
For any missing field, use null.

Required fields:
- name (string)
- brand (string)
- price_inr (number)
- category (string)
- in_stock (boolean)
- rating (number, 0-5, or null if not mentioned)

<product_text>
Mamaearth Vitamin C Face Wash 100ml is available on Nykaa for ₹249.
Currently in stock. Customers have rated it 4.3 out of 5.
</product_text>
```
**Expected response:**
```json
{
  "name": "Mamaearth Vitamin C Face Wash 100ml",
  "brand": "Mamaearth",
  "price_inr": 249,
  "category": "Face Wash",
  "in_stock": true,
  "rating": 4.3
}
```
No markdown. No 'Here is the JSON:'. No backticks. Machine-readable output.

---

## JSON + Chain-of-Thought Combined

You can embed reasoning *inside* a JSON field — the best of both worlds:

```
Classify this support ticket. Return ONLY a valid JSON object.

JSON schema:
{
  "category": "BILLING | DELIVERY | TECHNICAL | ACCOUNT | OTHER",
  "priority": "LOW | MEDIUM | HIGH | URGENT",
  "reasoning": "one sentence explaining your classification",
  "suggested_team": "string"
}

<ticket>
My account has been charged ₹4,999 twice for the same annual subscription.
I need this fixed immediately — I'm a premium user.
</ticket>
```
**Expected response:**
```json
{
  "category": "BILLING",
  "priority": "HIGH",
  "reasoning": "Duplicate charge on a premium subscription requires urgent financial correction.",
  "suggested_team": "Billing & Payments"
}
```

---

## 🔁 Quick Recap — Part 5

| Technique | When to Use |
|-----------|------------|
| JSON output prompting | When your code needs to parse the AI's response |
| Prefilling with `{` | Claude-specific — forces JSON mode instantly |
| `reasoning` field in JSON | Want explainability without ruining structure |
| Null for missing fields | Always specify this — prevents missing key errors |

## 🧪 Try This Yourself
Take any structured document you work with (an invoice, a job description, a meeting agenda). Write a JSON extraction prompt for it. Specify every field, its type, and what to do if the field is missing.

---
# 🚀 Part 6: Advanced Techniques

Three powerful techniques to know — explained simply.

---

## Technique 1: Self-Consistency (Wang et al. 2022)

**The Idea:** Run the *same prompt* multiple times. Take the most common answer.

**Why it works:** LLMs have some randomness (controlled by a setting called 'temperature'). Different runs produce slightly different reasoning paths. The *majority vote* answer is far more reliable than any single run.

**The Research:** Wang et al. (2022) showed self-consistency improved performance on the GSM8K math benchmark by **+17.9 percentage points** over single-run chain-of-thought.

**Practical Example:**
```
[Run this prompt 5 times, take the answer that appears most often]

A train covers 480 km at 80 km/h, then 360 km at 90 km/h.
What is the average speed for the whole journey?

Show your working.
```
Running this 5 times might give: 84.7, 84.7, 84.7, 86.4, 84.7 — majority answer: **84.7 km/h**

**When to use it in practice:**
```
You are analysing whether our product team's weekly all-hands meeting
should be shortened from 60 to 30 minutes.

Arguments for shortening:
- Team surveys show 60% find it too long
- 3 out of 5 managers say decisions happen in smaller groups anyway

Arguments against:
- Only meeting where all 45 people are in one room
- Culture and morale benefit mentioned by 40% of team

Make a recommendation. Let's think step by step.

[Run 3x. If 2+ agree, that's your answer.]
```

---

## Technique 2: Prompt Chaining

**The Idea:** Break one complex task into a *chain* of smaller prompts, where each step feeds the next.

**Why it works:** Doing everything in one giant prompt overwhelms the model. Small, focused steps produce better quality at each stage.

**Example Chain — Competitor Research to Content:**

```
Prompt 1:
List the top 5 competitors of Razorpay in the Indian payment gateway market.
For each, give their top 2 differentiators. Output as JSON.
         ↓
Prompt 2 (feeds Prompt 1 output):
Based on this competitor analysis: [PASTE JSON]
What are Razorpay's 3 strongest differentiators that competitors DON'T offer?
         ↓
Prompt 3 (feeds Prompt 2 output):
Write a 280-character tweet for Razorpay's LinkedIn page that highlights
this differentiator: [PASTE TOP DIFFERENTIATOR]
Target audience: Indian fintech startup founders.
         ↓
Prompt 4:
Expand that tweet into a 400-word LinkedIn post. Use a hook, 3 short
paragraphs, and end with a question to drive engagement.
```

**The result:** A high-quality, coherent LinkedIn post — built in focused steps rather than one unreliable mega-prompt.

---

## Technique 3: ReAct — Reason + Act

**The Idea:** Combine reasoning with *real-world actions* like web search, code execution, or database lookups. The AI thinks, then acts, then observes the result, then thinks again.

**Format: [Thought] → [Action] → [Observation] → repeat → [Final Answer]**

**Example — Reliance Stock Price Analysis:**
```
[Thought] The user wants to know if Reliance stock is up or down this week
          and why. I need to find the current price and recent news.

[Action] search: 'Reliance Industries stock price this week'

[Observation] Reliance stock is at ₹2,847, up 3.2% this week.

[Thought] Now I need the reason behind the rise.

[Action] search: 'Reliance Industries news this week 2025'

[Observation] Reliance announced a new JV with Saudi Aramco for green
              energy. Market reacted positively.

[Final Answer] Reliance Industries is up 3.2% this week to ₹2,847,
driven by positive sentiment around a newly announced green energy joint
venture with Saudi Aramco.
```

> **Note for beginners:** You don't need to write ReAct loops manually. Claude Skills (Part 7) handle this automatically — when Claude uses a web search skill, it's essentially running a ReAct loop behind the scenes.

---

## 🔁 Quick Recap — Part 6

| Technique | When to Use | Complexity |
|-----------|------------|-----------|
| Self-Consistency | High-stakes decisions, maths reasoning | Run N times, vote |
| Prompt Chaining | Multi-step workflows, research-to-content | Build a pipeline |
| ReAct | Tasks needing real-world data | Handled by Skills |

## 🧪 Try This Yourself
Take a multi-step task you do manually every week (research + summarise + draft + send). Map it as a 3-step prompt chain. Write all three prompts where step 2 clearly uses the output of step 1.

---
# 🛠️ Part 7: Claude Skills

## What LLMs Can't Do Without Help

Even the best-prompted LLM has hard limits. Without external tools, it:

| Limitation | Why |
|-----------|-----|
| ❌ Has no internet access | Training data has a cutoff date |
| ❌ Can't do precise maths | It predicts tokens, doesn't calculate |
| ❌ Has no memory between sessions | Each conversation starts fresh |
| ❌ Can't read your files | It only sees what you type |
| ❌ Can't take actions | It generates text, doesn't click or submit |

**Skills solve all of this.**

---

## What Are Claude Skills?

**Claude Skills** are reusable tool definitions that give Claude the ability to interact with the real world — searching the web, running code, reading files, calling APIs.

**Key dates:**
- 🗓️ **October 2025:** Anthropic officially launched Claude Skills
- 🗓️ **December 2025:** Anthropic made Skills an open standard (via the Model Context Protocol)

> 📖 Official announcement: https://www.anthropic.com/news/skills

---

## The Restaurant Chef Analogy

Think of Claude as a head chef:

```
Without Skills:  The chef knows every recipe in the world, but has no kitchen,
                 no ingredients, no oven. Brilliant knowledge, no execution.

With Skills:     The chef has a fully equipped kitchen — sharp knives (web search),
                 an oven (code execution), a walk-in fridge (file access),
                 a phone to call suppliers (APIs). Now they can actually cook.
```

---

## How a Skill Call Works

```
USER asks Claude a question
         ↓
Claude decides a Skill is needed
         ↓
Claude issues a 'tool call' (structured request to the skill)
         ↓
The SYSTEM executes the skill (searches web, runs code, reads file)
         ↓
RESULT is returned to Claude
         ↓
Claude incorporates the result into its final REPLY
         ↓
USER reads the answer
```

The entire loop is invisible to the user — they just asked a question and got a great answer.

---

## Real Examples

### Example 1: Web Search Skill — Morning Business Briefing

**Without Skills:**
```
User: What happened with Zepto's funding this week?
Claude: I don't have access to real-time information. My training data has
        a cutoff and I cannot tell you about recent news.
```

**With Web Search Skill:**
```
User: What happened with Zepto's funding this week?
[Claude triggers web_search skill]
[Skill searches: 'Zepto funding 2025 this week']
[Skill returns: 3 news articles from this week]
Claude: Zepto announced a $340M Series G funding round this week, valuing
        the company at $5B. The round was led by General Catalyst...
```

### Example 2: Code Execution Skill — Data Analysis

```
User: Here's our sales CSV for Q1. Calculate revenue by city and show me
      which city has the highest growth vs Q4 last year. [uploads CSV]

[Claude triggers code_execution skill]
[Skill runs Python: pandas analysis, calculates growth rates]
[Skill returns: dataframe with results]

Claude: Based on your Q1 data:
- Mumbai: ₹2.4Cr revenue, +34% vs Q4
- Bangalore: ₹1.8Cr revenue, +28% vs Q4
- Delhi: ₹1.6Cr revenue, +12% vs Q4

Mumbai is your fastest-growing city at +34% quarter-over-quarter.
```

---

## 5 Skill Types

| Skill Type | What It Does | Example Use Case |
|-----------|-------------|-----------------|
| **Web Search** | Searches the internet in real time | News briefings, competitor monitoring |
| **Code Execution** | Runs Python, analyses data, creates charts | CSV analysis, calculations |
| **File Reading** | Reads PDFs, Word docs, spreadsheets | Contract review, document Q&A |
| **MCP/APIs** | Calls external services (CRM, Slack, databases) | Salesforce lookup, GitHub integration |
| **Memory** | Stores and retrieves information between sessions | User preferences, project context |

---

## 🔁 Quick Recap — Part 7

| Without Skills | With Skills |
|---------------|------------|
| Static knowledge only | Real-time web access |
| No calculations | Code execution |
| No file reading | PDF, CSV, Word access |
| No external systems | API and MCP integrations |
| Forgets everything | Persistent memory |

## 🧪 Try This Yourself
Open Claude.ai. Enable web search from the tools menu. Ask: *'What are the top 3 news stories about Indian startups today? Summarise each in 2 sentences.'* Compare the response to what you get without the tool enabled.

---
# 📄 Part 8: The SKILL.md File

## What Is a SKILL.md File?

A **SKILL.md file** is a simple text file (written in Markdown) that describes a skill to Claude — telling it what the skill does, when to use it, and how to use it.

It lives inside a folder called your **skill folder.** The folder + the SKILL.md file together form a complete skill that Claude can understand and use.

Think of it like an employee onboarding document — except instead of onboarding a human, you're onboarding Claude to a new capability.

---

## Anatomy of a SKILL.md File

A SKILL.md has two main sections:

```
--- ← START of YAML frontmatter
name: skill-name
description: >
  A clear, specific description of what this skill does and WHEN to use it.
  This is the most important part — Claude reads this to decide whether to
  load and use the skill.
--- ← END of YAML frontmatter

# Main Content (Markdown body)
Everything below the --- is markdown that explains how the skill works,
what scripts it contains, and how to use them.
```

---

## Complete Example: Company Email Formatter Skill

```markdown
---
name: company-email-formatter
description: >
  Formats internal emails and external communications to follow Infosys
  brand guidelines. Use this skill whenever the user asks to write, revise,
  or improve a company email, client communication, or official letter.
  Also trigger for phrases like 'write an email to', 'draft a message to',
  or 'help me respond to this email'.
---

# Company Email Formatter Skill

Formats emails according to Infosys internal communication standards.

## When to Use
- User asks to write or improve any work email
- User pastes a received email and asks for a reply
- User asks to 'make this more professional'

## Rules
- Subject line: max 8 words, title case
- Opening: 'Dear [Name],' — never 'Hey' or 'Hi [Name],'
- Tone: professional, concise, respectful
- Signature block: Name | Role | Infosys | +91-XXXXXXXXXX
- Max length: 150 words for internal, 250 words for external

## Format
Output the email only — no commentary before or after.
```

---

## The #1 Rule: Write a Great `description`

> **Claude tends to under-trigger skills** — it won't use a skill unless the description clearly matches what the user is asking. The description is your skill's most important line.

### 🔴 Weak Description (will under-trigger)
```yaml
description: >
  Helps with PDF files.
```
**Problem:** 'Helps with PDF files' is too vague. Claude won't know *when* to use it.

### 🟢 Strong Description (will trigger reliably)
```yaml
description: >
  Use this skill to read, extract text from, or answer questions about PDF
  files uploaded by the user. Triggers when: the user uploads a .pdf file,
  asks to 'read this document', 'summarise this PDF', 'find X in this file',
  or uses phrases like 'based on this document'. Also triggers for any
  contract review, invoice analysis, or report summarisation task where
  a PDF may be involved.
```
**Why it's better:** Specific trigger phrases, specific use cases, covers edge cases.

---

## Skill Folder Structure

```
support-ticket-triage/
├── SKILL.md              ← The main descriptor file (required)
├── scripts/
|   └── classify.py       ← Python script for classification logic
├── references/
|   └── categories.md     ← Reference list of ticket categories
└── assets/
    └── examples.json     ← Example tickets for few-shot context
```

---

## Progressive Loading Explained

Claude doesn't load every SKILL.md into memory at once (that would be expensive and slow). Instead:

1. Claude reads **only the `description`** field first to decide if the skill is relevant
2. If relevant, Claude loads the **full SKILL.md body**
3. If the skill body references scripts, Claude loads **only the scripts it needs**

**This is why the description must be excellent** — it's the only part Claude reads before deciding whether to use the skill at all.

---

## Full Recipe: Build a Support-Ticket-Triage Skill from Scratch

**Step 1:** Create the folder
```bash
mkdir support-ticket-triage
cd support-ticket-triage
```

**Step 2:** Write the SKILL.md
```markdown
---
name: support-ticket-triage
description: >
  Classifies and prioritises incoming customer support tickets.
  Use when the user pastes a support ticket, complaint email, or customer
  message and asks to 'classify this', 'prioritise this', 'route this ticket',
  or 'what team should handle this'. Also trigger for batch analysis of
  multiple tickets.
---

# Support Ticket Triage Skill

## Classification Schema
| Category | Examples |
|----------|---------|
| BILLING | Duplicate charges, refund requests, invoice errors |
| DELIVERY | Late delivery, wrong item, missing item |
| TECHNICAL | App crash, login issues, payment failure |
| ACCOUNT | Password reset, account locked, verification |
| OTHER | General feedback, compliments, unclassifiable |

## Priority Rules
- URGENT: Financial loss > ₹500 OR account locked
- HIGH: Financial loss ≤ ₹500 OR delivery delay > 48 hours
- MEDIUM: Technical issues without financial impact
- LOW: General feedback, questions, feature requests

## Output Format
Return a JSON object with: category, priority, reasoning, suggested_team
```

**Step 3:** Test it in Claude.ai
```bash
Prompt : 'Classify this: My account was
charged ₹999 twice for the same order. Fix this immediately.'
```

---

## Where to Use Skills

| Platform | How to Add Skills |
|---------|------------------|
| **Claude.ai** | Project Settings → Add Skill → Upload folder |
| **Claude Code** | `~/.claude/skills/` folder — auto-loaded |
| **API** | Pass SKILL.md content in system message or via tool definitions |

---

## SKILL.md vs System Prompt

| | System Prompt | SKILL.md |
|-|--------------|---------|
| **Scope** | Whole conversation | One specific capability |
| **Trigger** | Always active | Only when description matches |
| **Reusability** | Per conversation | Shareable across projects |
| **Best For** | Persona, tone, global rules | Tools, specialised tasks |
| **Format** | Free text | YAML frontmatter + Markdown |

**Rule of thumb:** System prompt = who Claude is. SKILL.md = what Claude can do.

---

## 🔁 Quick Recap — Part 8

| Concept | Key Point |
|---------|----------|
| SKILL.md | A text file that onboards Claude to a new capability |
| Description | The MOST important field — Claude reads this first |
| Under-triggering | Claude's #1 skill failure mode — fix with specific trigger phrases |
| Folder structure | SKILL.md + scripts/ + references/ + assets/ |
| Progressive loading | Description first, then body, then scripts |

## 🧪 Try This Yourself
Create a SKILL.md for a skill you'd genuinely find useful — maybe a 'meeting notes formatter' or a 'code review checker'. Write the YAML frontmatter with a strong description, then add 3–5 rules in the markdown body.

---
# ⚖️ Part 9: Prompting vs Skills — When to Use What

## The Full Comparison

| Dimension | Prompting | Skills |
|-----------|-----------|--------|
| **What it is** | Text instructions to Claude | Code + descriptor files |
| **Lives where** | In your message / system prompt | In a folder on disk |
| **Can access internet** | ❌ No | ✅ Yes (web search skill) |
| **Can run code** | ❌ No | ✅ Yes (code execution skill) |
| **Can read files** | ❌ No | ✅ Yes (file reading skill) |
| **Learning curve** | Low | Medium |
| **Best for** | Language tasks, analysis, writing | Actions, data, real-world tasks |
| **Shareable** | Copy-paste | Version control, git |

---

## Decision Framework

```
Is your task purely about language — writing, analysis, classification, summarisation?
    YES → Use Prompting
    NO  ↓

Does your task need real-time data, file access, or code execution?
    YES → Use Skills
    NO  ↓

Is this task repetitive and reusable across many contexts?
    YES → Build a Skill (even if it's just prompting logic wrapped in SKILL.md)
    NO  → Use Prompting
```

---

## Ultimate Combined Example: Sales Manager Risk Analysis

Imagine a sales manager at an Indian B2B software company wants a risk analysis for a large deal. Here's how all the techniques combine:

**System Prompt (Prompting):**
```
You are a senior B2B sales strategist with 15 years of experience in enterprise
software deals in India. You are analytical, direct, and never sugarcoat risk.
```

**CRM Skill (retrieves deal data):**
```
[CRM Skill retrieves:]
Deal: TechCorp Mumbai | Value: ₹45L | Stage: Proposal Sent | Days in stage: 21
Contact: Vikram Mehta (CTO) | Last interaction: 14 days ago
Competitors: Known evaluation with SAP and a local vendor
```

**Web Search Skill (retrieves market intelligence):**
```
[Web Search Skill finds:]
Recent news: TechCorp announced a hiring freeze last week
IT budget cuts: 3 articles suggest Indian mid-market IT spending down 12% in Q3
```

**Prompt (Chain-of-Thought + Few-Shot voice match):**
```
Based on the CRM data and market intelligence above, analyse the risk of this deal.

Think step by step:
1. Assess deal health signals
2. Assess market context risks
3. Rate overall deal risk (LOW / MEDIUM / HIGH / CRITICAL)
4. Recommend exactly 2 actions the sales manager should take this week

Match this communication style:
[EXAMPLE] 'This deal is at risk. Vikram's gone quiet, SAP is circling,
and the CFO wasn't in the last call. Either get a meeting by Friday or
qualify it out. Don't waste another sprint on a ghost.'
```

**Expected response:**
```
1. DEAL HEALTH: Danger signs everywhere. 21 days stalled in Proposal stage is 2x
   the healthy norm. 14 days since last contact with the CTO is a red flag —
   either he's disengaged or the decision has moved to a committee you're not in.

2. MARKET CONTEXT: The hiring freeze announcement is a significant budget signal.
   Combined with sector-wide IT spending down 12%, TechCorp may be looking for
   reasons to defer, not approve. SAP has deeper enterprise relationships here.

3. RISK RATING: HIGH

4. TWO ACTIONS THIS WEEK:
   → ACTION 1: Send Vikram a 'value confirmation' email today — not a follow-up,
     but a one-pager tying your solution directly to a cost they can cut now.
     Reframe as savings, not investment, given the freeze.
   → ACTION 2: Find a second contact inside TechCorp — Finance or VP Ops —
     who has budget visibility. You're currently too dependent on a single
     champion who's gone quiet.
```

This response used: **Role assignment + CRM Skill + Web Search Skill + Chain-of-Thought + Few-Shot voice matching** — all working together.

---

## 🔁 Quick Recap — Part 9

**The simplest rule:** Start with prompting. When you hit a wall — real-time data, file access, code, or automation — add a skill.

---
# 🏁 Big Recap — Everything We Covered Today

| Part | Topic | The One Thing to Remember |
|------|-------|--------------------------|
| 1 | Anatomy of a Prompt | System (handbook) + User (task) + Assistant (history) |
| 2 | 6 Fundamentals | Clear → Role → XML → Format → 'I don't know' → Iterate |
| 3 | Few-Shot | Show 2–5 examples before your real question |
| 4 | Chain-of-Thought | 'Let's think step by step' or use structured steps |
| 5 | JSON Output | Specify schema + types + null policy + no backticks |
| 6 | Advanced Techniques | Self-consistency (vote), chaining (pipeline), ReAct (loop) |
| 7 | Claude Skills | Give Claude eyes, hands, and memory |
| 8 | SKILL.md | Great description = reliable triggering |
| 9 | Prompting vs Skills | Language tasks → prompt. Real-world tasks → skill. |

**Key Stats (Fact-Checked):**
- **Kojima et al. 2022:** 'Let's think step by step' → MultiArith 17.7% → 78.7%, GSM8K 10.4% → 40.7%
- **Wei et al. 2022:** Few-shot CoT on PaLM 540B → 58% on GSM8K
- **Wang et al. 2022:** Self-consistency → +17.9% on GSM8K over single-run CoT
- **Claude Skills launched:** October 2025 (open standard: December 2025)

---

# 📋 Cheat Sheet — Situation to Action

| Your Situation | Technique to Use |
|---------------|-----------------|
| AI gives inconsistent tone/format | Add a system message with persona + format rules |
| Response is too generic | Add role assignment: 'You are a senior X with Y years...' |
| Data is getting confused with instructions | Use XML tags `<data>`, `<instructions>` |
| AI adds unwanted commentary to JSON | 'Return ONLY valid JSON. No preamble, no backticks.' |
| AI makes up facts | Add 'If you don't know, say I don't have reliable data.' |
| Output format is wrong every time | Specify exact format with an example |
| Need consistent style/voice matching | Use few-shot examples (2–5) |
| Maths or logic errors | Add 'Let's think step by step' or use structured CoT |
| High-stakes decision, need reliability | Run prompt 3–5x, take majority answer (self-consistency) |
| Task is too complex for one prompt | Break into a 3–4 step prompt chain |
| Need real-time information | Enable web search skill |
| Need to analyse a CSV or run calculations | Enable code execution skill |
| Need to read a PDF or document | Enable file reading skill |
| Need Claude to remember across sessions | Use memory skill |
| Building a reusable AI tool | Create a SKILL.md with a strong description |
| On a reasoning model (o1, Claude ET) | Skip CoT — model already reasons internally |

---

# 🏆 Final Challenge

Before Day 4, complete this challenge:

**Part A — Write the Perfect Prompt**
Choose a real task from your job (report writing, ticket classification, email drafting, data analysis). Apply at least 4 of the 6 fundamentals. Include a role, XML tags, specified format, and an escape hatch for 'I don't know'.

**Part B — Build Your First SKILL.md**
Create a SKILL.md for the same task. Write a description with at least 5 specific trigger phrases. Add 3 rules the skill must follow. Test it in Claude.ai.

**Share it in the WhatsApp group before Sunday's session.** The best examples will be reviewed live.

---

# 📚 Further Reading & Official Sources

| Resource | Link |
|----------|------|
| Anthropic Official Prompt Engineering Docs | https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/overview |
| Claude Skills Official Announcement | https://www.anthropic.com/news/skills |
| Wei et al. 2022 — Chain-of-Thought Paper | https://arxiv.org/abs/2201.11903 |
| Kojima et al. 2022 — Zero-Shot CoT Paper | https://arxiv.org/abs/2205.11916 |
